In [26]:
import pandas as pd
from mortgage_default_risk.loader import COLUMNS
from mortgage_default_risk.loader import find_file,read_sample,count_non_empty_values

In [4]:
rows = 1000
q = "2022Q1"
file = find_file(q)
data = read_sample(q,rows)


In [5]:
data.head()

,reference_pool_id,loan_id,reporting_period,channel,seller_name,servicer_name,master_servicer,orig_interest_rate,current_interest_rate,orig_upb,...,repurchase_make_whole_proceeds_flag,alt_delinquency_resolution,alt_delinquency_resolution_count,total_deferral_amount,payment_deferral_modification_flag,interest_bearing_upb,orig_classic_fico,orig_vantagescore_4,pricing_indicator,orig_fannie_score
0,NaN,000130357804,012022,R,Other,"Lakeview Loan Servicing, LLC",NaN,3.375,3.375,255000.00,...,NaN,7,NaN,NaN,7,NaN,NaN,NaN,NaN,NaN
1,NaN,000130357804,022022,R,Other,"Lakeview Loan Servicing, LLC",NaN,3.375,3.375,255000.00,...,NaN,7,NaN,NaN,7,NaN,NaN,NaN,NaN,NaN
2,NaN,000130357804,032022,R,Other,"Lakeview Loan Servicing, LLC",NaN,3.375,3.375,255000.00,...,NaN,7,NaN,NaN,7,NaN,NaN,NaN,NaN,NaN
3,NaN,000130357804,042022,R,Other,"Lakeview Loan Servicing, LLC",NaN,3.375,3.375,255000.00,...,NaN,7,NaN,NaN,7,NaN,NaN,NaN,NaN,NaN
4,NaN,000130357804,052022,R,Other,"Lakeview Loan Servicing, LLC",NaN,3.375,3.375,255000.00,...,NaN,7,NaN,NaN,7,NaN,NaN,NaN,NaN,NaN


In [7]:
non_empty = count_non_empty_values(q,100000)

In [13]:
total_non_empty = non_empty[0]
total_rows = non_empty[1]
percentages = ((total_non_empty/total_rows)*100).round(1)

In [25]:
columns_percent_zero = []
for column,percentage in percentages.items():
    if percentage == 0.0:
        columns_percent_zero.append(column)
print(columns_percent_zero)

['reference_pool_id', 'master_servicer', 'upb_at_issuance', 'io_first_principal_and_interest_payment_date', 'months_to_amortization', 'mi_cancellation_flag', 'repurchase_date', 'scheduled_principal_current', 'unscheduled_principal_current', 'last_paid_installment_date', 'foreclosure_date', 'disposition_date', 'foreclosure_costs', 'property_preservation_repair_costs', 'asset_recovery_costs', 'misc_holding_expenses_credits', 'holding_property_taxes', 'net_sales_proceeds', 'credit_enhancement_proceeds', 'repurchase_make_whole_proceeds', 'other_foreclosure_proceeds', 'orig_list_start_date', 'orig_list_price', 'current_list_start_date', 'current_list_price', 'borrower_credit_score_issuance', 'co_borrower_credit_score_issuance', 'borrower_credit_score_current', 'co_borrower_credit_score_current', 'current_period_modification_loss', 'cumulative_modification_loss', 'current_period_credit_event_net_gain_loss', 'cumulative_credit_event_net_gain_loss', 'zero_balance_code_change_date', 'loan_holdb

In [45]:
def check_fico(q,chunksize):
    file = find_file(q)
    reader = pd.read_csv(file,header=None,sep="|",names=COLUMNS,chunksize=chunksize,usecols=["loan_id", "reporting_period", "orig_classic_fico", "borrower_credit_score_orig"])
    months_counts = pd.Series(dtype="int64")
    score_counts = pd.Series(dtype="int64")
    samples = []

    for chunk in reader:
        mask = chunk["orig_classic_fico"].notna()
        fico_true = chunk[mask]

        months_counts = months_counts.add(fico_true["reporting_period"].value_counts(),fill_value=0)
        score_counts = score_counts.add(fico_true["orig_classic_fico"].value_counts(),fill_value=0)

        if not samples and not fico_true.empty:
            samples.append(fico_true.head(20))
    sample = samples[0] if samples else pd.DataFrame()
    return months_counts.astype(int), score_counts.astype(int), sample


In [47]:
months,scors,sample = check_fico("2022Q1",500_00)

In [48]:
months.index = pd.to_datetime(months.index, format="%m%Y")
months.sort_index()

C:\Users\sasi1\AppData\Local\Temp\ipykernel_38824\2905819214.py:1: Pandas4Warning: Parsing integer or float values with a format in to_datetime is deprecated. In a future version these will be interpreted as epochs via the 'unit' keyword instead. Cast them to strings first to retain the current behavior.
  months.index = pd.to_datetime(months.index, format="%m%Y")


reporting_period
2025-12-01    674420
2026-01-01    671473
2026-02-01    669211
2026-03-01    666821
2026-04-01    663917
2026-05-01    660753
2026-06-01    657419
dtype: int64

In [49]:
score_values = scores.index.astype(int)
score_values.min(), score_values.max()

(np.int64(441), np.int64(832))

In [50]:
sample[["loan_id", "reporting_period", "borrower_credit_score_orig", "orig_classic_fico"]]

,loan_id,reporting_period,borrower_credit_score_orig,orig_classic_fico
47,130357804,122025,703.0,703.0
48,130357804,12026,703.0,703.0
49,130357804,22026,703.0,703.0
50,130357804,32026,NaN,703.0
51,130357804,42026,NaN,703.0
52,130357804,52026,NaN,703.0
53,130357804,62026,NaN,703.0
101,130357805,122025,691.0,691.0
102,130357805,12026,691.0,691.0
103,130357805,22026,691.0,691.0
